---
title: Venues & publishers
description: Where the School of Electrical and Computer Engineering of the Aristotle University of Thessaloniki publishes – publisher groups over time, the most-used journals and conference series, citation impact by venue and the diversity of venues, from Scopus data since 1981.
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "sece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sece_stats import config, data
from sece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the export, partial
AS_OF = config.EXPORT_DATE      # the date of the Scopus export

In [ ]:
import math
import re
import unicodedata

import plotly.graph_objects as go
from sece_stats import metrics, periods

pubs = data.load_publications()
n_pubs = len(pubs)
FIRST_YEAR = int(pubs["year"].min())
LAST_FULL = periods.LAST_FULL                 # the latest full year (PARTIAL - 1)
AS_OF_TEXT = f"{AS_OF.day} {AS_OF:%B %Y}"     # the export date as "day month year"
assert FIRST_YEAR == 1981, "description (frontmatter): 'from Scopus data since 1981'"

# Five-year periods that end with the latest full year (the site's shared rule): the first one is
# clipped to the data, and the partial year has a label of its own outside the periods.
pubs["period"] = pubs["year"].map(lambda year: periods.period_of(year, first_year=FIRST_YEAR))
is_full = pubs["year"] <= LAST_FULL
PERIODS = periods.periods(FIRST_YEAR)            # oldest first; the last one is P_RECENT
P_RECENT = periods.period_of(LAST_FULL)          # the latest five full years
P_PREV = periods.period_of(LAST_FULL - 5)        # the five years before them
P_BASE = periods.period_of(LAST_FULL - 20)       # four periods earlier: the base of the long comparisons
P_STABLE = periods.period_of(LAST_FULL - 15)     # the period after P_BASE
MIN_PERIOD_N = 100                               # fewest records of a period in the period comparisons
RECENT_COL = f"{periods.bounds(P_RECENT)[0]}–{periods.bounds(P_RECENT)[1] % 100:02d}"   # P_RECENT as a table header
in_recent = pubs["period"].eq(P_RECENT)
in_prev = pubs["period"].eq(P_PREV)


def fold(text: object) -> str:
    """Lower-case ASCII text without accents ('' for missing values)."""
    if not isinstance(text, str):
        return ""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch)).lower()


def title_key(text: object) -> str:
    """Matching key of a title: folded, '&' -> 'and', punctuation removed."""
    key = re.sub(r"[^0-9a-z]+", " ", fold(text).replace("&", " and "))
    return re.sub(r"\s+", " ", key).strip()


def modal(values: pd.Series):
    """Most frequent value (ties: the first in order of appearance; None when empty)."""
    counts = values.value_counts(sort=False)
    return counts.idxmax() if len(counts) else None


def join_and(items: list[str]) -> str:
    """'a', 'a and b', 'a, b and c'."""
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


# ------------------------------------------------------------------------------ publishers
# Raw Scopus "Publisher" values -> publisher group (the current owner of the imprint);
# first matching rule wins, other publishers keep their own name.
PUBLISHER_RULES = [
    ("Wiley", r"wiley-ieee"),
    ("IEEE", r"\bieee\b|institute of electrical and electronics engineers"),
    ("IET", r"institution of engineering and technology|^iee$|institution of electrical engineers|"
            r"peter peregrinus"),
    ("Elsevier", r"elsevier|academic press|pergamon|north[- ]holland"),
    ("Springer Nature", r"springer|kluwer academic|nature research|nature publishing|biomed central|"
                        r"birkhauser"),
    ("MDPI", r"\bmdpi\b|multidisciplinary digital publishing"),
    ("Wiley", r"wiley|blackwell|hindawi"),
    ("Taylor & Francis", r"taylor (?:and|&) francis|routledge|crc press"),
    ("AIP Publishing", r"american institute of physics"),
    ("IOP Publishing", r"institute of physics|\biop\b"),
    ("ACM", r"association for computing machinery"),
    ("American Physical Society", r"american physical society"),
    ("Optica", r"optical society|\bosa\b|optica publishing"),
    ("SPIE", r"^spie$"),
    ("Frontiers", r"^frontiers"),
    ("SciTePress", r"scitepress|science and technology publications, lda"),
    ("Oxford University Press", r"oxford univ"),
    ("Cambridge University Press", r"cambridge univ"),
    ("World Scientific", r"^world scientific publishing|^world scientific$"),
    ("Emerald", r"^emerald"),
    ("IOS Press", r"^ios press"),
    ("SAGE", r"\bsage\b"),
    ("De Gruyter", r"de gruyter"),
    ("Wolters Kluwer", r"lippincott"),
    ("VDE Verlag", r"^vde verlag"),
    # smaller publishers whose Scopus names vary, under one name (the name the DOI rule below uses)
    ("ASME", r"american society of mechanical engineers|\basme\b"),
    ("IFAC", r"\bifac\b|international federation of automatic control"),
    ("Inderscience", r"^inderscience"),
    ("JMIR Publications", r"^jmir publications"),
    ("MIT Press", r"^mit press"),
    ("SIAM", r"society for industrial and applied mathematics|^siam\b"),
    ("Thieme", r"\bthieme\b"),
    ("BMJ", r"^bmj\b"),
    ("Multi-Science Publishing", r"^multi-science"),
    ("Editrice Compositori", r"^editrice compositori"),
    ("Polish Academy of Sciences", r"polish academy of sciences|polska akademia nauk"),
    ("Copernicus Publications", r"^copernicus"),
    ("Korean Institute of Communications and Information Sciences", r"^korean? institute of communications?\b"),
    ("International Society of Automation", r"^isa\b|instrument society of america"),
    ("INSTICC", r"\binsticc\b"),
    ("European Space Agency", r"european space agency|^esa\b"),
]
PUBLISHER_RX = [(group, re.compile(rx)) for group, rx in PUBLISHER_RULES]

# DOI prefixes (the DOI registrant) -> publisher group, used when the Publisher field is empty;
# the names are those that publisher_group() gives to the Publisher values of the same publishers.
DOI_PREFIX_GROUP = {
    "10.1109": "IEEE", "10.23919": "IEEE", "10.1049": "IET",
    "10.1016": "Elsevier", "10.1006": "Elsevier",
    "10.1007": "Springer Nature", "10.1023": "Springer Nature", "10.1038": "Springer Nature",
    "10.1186": "Springer Nature", "10.1140": "Springer Nature", "10.2200": "Springer Nature",
    "10.3390": "MDPI",
    "10.1002": "Wiley", "10.1111": "Wiley", "10.1155": "Wiley", "10.1100": "Wiley",
    "10.1080": "Taylor & Francis", "10.1201": "Taylor & Francis", "10.3109": "Taylor & Francis",
    "10.1179": "Taylor & Francis",
    "10.1063": "AIP Publishing", "10.1088": "IOP Publishing", "10.1145": "ACM",
    "10.1103": "American Physical Society", "10.1364": "Optica", "10.1117": "SPIE",
    "10.3389": "Frontiers", "10.5220": "SciTePress", "10.1093": "Oxford University Press",
    "10.1017": "Cambridge University Press", "10.1142": "World Scientific", "10.1108": "Emerald",
    "10.3233": "IOS Press", "10.1177": "SAGE", "10.1515": "De Gruyter", "10.2478": "De Gruyter",
    "10.1097": "Wolters Kluwer", "10.2528": "Electromagnetics Academy",
    "10.1371": "Public Library of Science", "10.1051": "EDP Sciences",
    "10.1121": "Acoustical Society of America", "10.1115": "ASME", "10.1162": "MIT Press",
    "10.1163": "Brill", "10.1556": "Akadémiai Kiadó", "10.2316": "ACTA Press", "10.4018": "IGI Global",
    "10.1504": "Inderscience", "10.3182": "IFAC", "10.2196": "JMIR Publications", "10.1055": "Thieme",
    "10.1137": "SIAM", "10.5755": "Kaunas University of Technology", "10.3144": "BME-PT",
    "10.1029": "American Geophysical Union", "10.1136": "BMJ", "10.1260": "Multi-Science Publishing",
    "10.3899": "The Journal of Rheumatology", "10.4401": "Editrice Compositori",
    "10.4156": "Advanced Institute of Convergence Information Technology",
}

# Venue names that identify the publisher (records with neither a publisher nor a DOI).
NAME_RULES = [
    ("IEEE", r"\bieee\b"),
    ("IET", r"^iee\b"),
    ("Springer Nature", r"^(?:lecture notes in|communications in computer and information science|"
                        r"ifip advances|ifmbe proceedings|mechanisms and machine science)"),
    ("CEUR-WS", r"^ceur workshop proceedings"),
    ("Optica", r"^optics infobase"),
    ("SPIE", r"^proceedings of spie"),
    ("IOS Press", r"^studies in health technology and informatics"),
    ("ACM", r"^acm international conference proceeding series"),
]
NAME_RX = [(group, re.compile(rx)) for group, rx in NAME_RULES]


def publisher_group(name: object) -> str | None:
    """Publisher group of a raw Publisher value (the raw name for small publishers)."""
    if not isinstance(name, str) or not name.strip():
        return None
    low = fold(name)
    return next((group for group, rx in PUBLISHER_RX if rx.search(low)), name.strip())


def name_group(*names: object) -> str | None:
    """Publisher implied by a source title or conference name, if any."""
    for name in names:
        low = fold(name)
        for group, rx in NAME_RX:
            if low and rx.search(low):
                return group
    return None


group = pubs["publisher"].map(publisher_group).astype(object)
source = pd.Series(np.where(group.notna(), "field", "unknown"), index=pubs.index, dtype=object)
# IET journals that Scopus lists under Wiley, the IET's publishing partner since 2021
iet_wiley = pubs["source_title"].str.match(r"^IET\b", na=False) & group.eq("Wiley")
n_iet_wiley = int(iet_wiley.sum())
group[iet_wiley] = "IET"
doi_prefix = pubs["doi"].str.extract(r"^\s*(10\.\d{4,9})/", expand=False)
by_doi = doi_prefix.map(DOI_PREFIX_GROUP)
take = group.isna() & by_doi.notna()
group[take], source[take] = by_doi[take], "doi"
assert by_doi[pubs["publisher"].isna() & pubs["doi"].notna()].notna().all(), \
    "notes: every record without a Publisher value that has a DOI is assigned from its prefix (extend DOI_PREFIX_GROUP)"
by_name = pd.Series([name_group(s, c) for s, c in zip(pubs["source_title"], pubs["conference_name"])],
                    index=pubs.index, dtype=object)
take = group.isna() & by_name.notna()
group[take], source[take] = by_name[take], "name"

# ------------------------------------------------------------------------------ journals
is_journal = pubs["source_kind"].eq("journal")
is_conf = pubs["source_kind"].eq("conference")
tkey = pubs["source_title"].map(title_key)
# Titles that share an ISSN are one journal (renamed journals, spelling variants).
_parent: dict[str, str] = {}


def _root(key: str) -> str:
    while _parent.setdefault(key, key) != key:
        key = _parent[key]
    return key


for _, titles in pubs.loc[is_journal & pubs["issn"].notna()].groupby("issn")["source_title"]:
    keys = sorted(set(titles.map(title_key)))
    for key in keys[1:]:
        _parent[_root(key)] = _root(keys[0])
jkey = tkey.map(_root)
n_merged_titles = int(tkey[is_journal].nunique() - jkey[is_journal].nunique())
# Renamed journals that received a new ISSN stay separate journals, such as the IEE Proceedings and
# the IET journals that continued them (same subject, "IEE Proceedings: X" -> "IET X").
_jkeys = set(jkey[is_journal])
iee_renamed = sorted(k for k in _jkeys if k.startswith("iee proceedings ")
                     and "iet " + k[len("iee proceedings "):] in _jkeys)
assert iee_renamed, "notes: renamed journals with a new ISSN (IEE Proceedings and their IET successors) count separately"

# last fallback for journals: the publisher of the journal's other records, when they agree
known = pd.DataFrame({"journal": jkey, "group": group})[is_journal & group.notna()]
journal_group = known.groupby("journal")["group"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
by_journal = jkey.map(journal_group)
take = group.isna() & is_journal & by_journal.notna()
group[take], source[take] = by_journal[take], "journal"
pubs["publisher_group"] = group.fillna("Unknown")

n_pub_missing = int(pubs["publisher"].isna().sum())
share_pub_missing = n_pub_missing / n_pubs
GAP_FROM, GAP_TO = 2000, 2014     # the years in which Scopus mostly lacks the Publisher value (fixed)
GAP_TEXT = f"{GAP_FROM}–{GAP_TO}"
in_gap = pubs["year"].between(GAP_FROM, GAP_TO)
share_pub_missing_gap = pubs.loc[in_gap, "publisher"].isna().mean()       # of the records of these years
missing_in_gap = (in_gap & pubs["publisher"].isna()).sum() / n_pub_missing   # of the missing values
assert missing_in_gap > 0.5, "text: most of the missing Publisher values are from GAP_TEXT"
n_by_doi, n_by_name, n_by_journal, n_unknown = (
    int(source.eq(k).sum()) for k in ("doi", "name", "journal", "unknown"))
share_unknown = n_unknown / n_pubs
unk_years = pubs.loc[source.eq("unknown"), "year"]
unk_first, unk_last = int(unk_years.min()), int(unk_years.max())
assert n_by_doi + n_by_name + n_by_journal + n_unknown == n_pub_missing, \
    "text: the missing publishers were filled from DOI, name and journal; the rest stay unknown"
n_unk_conf = int((source.eq("unknown") & is_conf).sum())   # conference papers (no unknown record has a DOI)
# checks of the fallbacks on the records that do have a Publisher value
has_field = source.eq("field")
chk = has_field & by_doi.notna()
n_doi_check, doi_agree = int(chk.sum()), (by_doi[chk] == pubs.loc[chk, "publisher_group"]).mean()
differ = chk & by_doi.ne(pubs["publisher_group"])
spelling = [(d, f) for d, f in zip(by_doi[differ], pubs.loc[differ, "publisher"])
            if re.search(rf"\b{re.escape(fold(d))}\b", fold(f)) or re.search(rf"\b{re.escape(fold(f))}\b", fold(d))]
assert not spelling, f"notes: the DOI check counts two spellings of one publisher as a disagreement: {spelling[:3]}"
chk = has_field & by_name.notna()
n_name_check, name_agree = int(chk.sum()), (by_name[chk] == pubs.loc[chk, "publisher_group"]).mean()
is_hindawi = (pubs["publisher"].str.contains("hindawi", case=False, na=False)
              | (source.eq("doi") & doi_prefix.isin(["10.1155", "10.1100"])))
n_hindawi = int(is_hindawi.sum())
assert pubs.loc[is_hindawi, "source_title"].str.contains("eurasip", case=False, na=False).any(), \
    "notes: the Hindawi records include EURASIP journals"

pub_counts = pubs.loc[pubs["publisher_group"].ne("Unknown"), "publisher_group"].value_counts()
TOP_PUBLISHERS = list(pub_counts.index[:6])           # the six largest groups, largest first
PUBLISHER_ORDER = TOP_PUBLISHERS + ["Other", "Unknown"]
PUBLISHER_COLORS = dp.color_map(PUBLISHER_ORDER)
pubs["publisher_fold"] = pubs["publisher_group"].where(
    pubs["publisher_group"].isin(TOP_PUBLISHERS + ["Unknown"]), "Other")
n_ieee = int(pub_counts["IEEE"])
share_ieee, share_els, share_spr = (pub_counts[p] / n_pubs for p in ("IEEE", "Elsevier", "Springer Nature"))

full = pubs[is_full]
per_counts = (pd.crosstab(full["period"], full["publisher_fold"])
              .reindex(index=PERIODS, columns=PUBLISHER_ORDER, fill_value=0))
per_n = per_counts.sum(axis=1)
per_share = per_counts.div(per_n, axis=0)
# The period comparisons start with P_FIRST, the first period with at least MIN_PERIOD_N records.
BIG_PERIODS = [p for p in PERIODS if per_n[p] >= MIN_PERIOD_N]
P_FIRST = BIG_PERIODS[0]
assert BIG_PERIODS == PERIODS[PERIODS.index(P_FIRST):], "text: only the earliest periods have few records"
ieee_first, ieee_prev, ieee_recent = (per_share.at[p, "IEEE"] for p in (P_FIRST, P_PREV, P_RECENT))
assert ieee_first < ieee_prev, "text: the IEEE share rose from P_FIRST to P_PREV"
ieee_major = (per_share["IEEE"] > 0.5).to_numpy()
assert ieee_major[-1], "text: IEEE published more than half of the output of P_RECENT"
ieee_major_from = PERIODS[max((i for i, m in enumerate(ieee_major) if not m), default=-1) + 1]   # majority ever since
els_first, els_base, els_recent = (per_share.at[p, "Elsevier"] for p in (P_FIRST, P_BASE, P_RECENT))
assert els_first > els_base > els_recent, "text: the Elsevier share fell from P_FIRST to P_BASE and P_RECENT"
spr_lo, spr_hi = per_share.loc[BIG_PERIODS, "Springer Nature"].agg(["min", "max"])
unk_max_period = per_share["Unknown"].idxmax()
unk_max = per_share["Unknown"].max()
assert unk_max < 0.1, "text: unknown publishers are too few to change the trends"
ieee_conf_recent, ieee_jour_recent = (
    pubs.loc[in_recent & pubs["source_kind"].eq(kind), "publisher_group"].eq("IEEE").mean()
    for kind in ("conference", "journal"))
assert ieee_conf_recent > ieee_jour_recent, "text: IEEE's lead is largest in conferences"

is_mdpi = pubs["publisher_group"].eq("MDPI")
mdpi_first = int(pubs.loc[is_mdpi, "year"].min())
mdpi_prev, mdpi_recent = per_share.at[P_PREV, "MDPI"], per_share.at[P_RECENT, "MDPI"]
n_mdpi_recent = int((is_mdpi & in_recent).sum())
mdpi_jour_recent = is_mdpi[in_recent & is_journal].mean()      # share of the journal papers
assert mdpi_recent > mdpi_prev, "text: the MDPI share grew from P_PREV to P_RECENT"
group_growth = per_share.loc[P_RECENT, TOP_PUBLISHERS] / per_share.loc[P_PREV, TOP_PUBLISHERS]
assert group_growth.idxmax() == "MDPI", "text: MDPI is the fastest-growing of the six largest groups"
mdpi_year = is_mdpi[is_full].groupby(pubs.loc[is_full, "year"]).mean()
mdpi_peak_year = int(mdpi_year.idxmax())
mdpi_peak, mdpi_last_full = mdpi_year.max(), mdpi_year[LAST_FULL]


def mdpi_annual() -> str:
    """MDPI's annual share from its peak to the latest full year (direction from the data)."""
    after = mdpi_year.loc[mdpi_peak_year + 1:]
    if after.empty:
        return f"was highest in {LAST_FULL}, at {dp.fmt_pct(mdpi_peak)}"
    low_year = int(after.idxmin())
    peak = f"peaked at {dp.fmt_pct(mdpi_peak)} in {mdpi_peak_year}"
    if low_year == LAST_FULL:
        return f"{peak} before falling to {dp.fmt_pct(mdpi_last_full)} in {LAST_FULL}"
    return (f"{peak}, fell to {dp.fmt_pct(after[low_year])} in {low_year} and was "
            f"{dp.fmt_pct(mdpi_last_full)} in {LAST_FULL}")


mdpi_annual_text = mdpi_annual()

# ------------------------------------------------------------------------------ conference series
# Containers that publish many unrelated conferences (book series, proceedings services):
# a conference paper with no conference name in one of these has no series.
CONTAINER_RX = re.compile(
    r"^(?:lecture notes|communications in computer and information science|ceur workshop|"
    r"proceedings of spie|iet conference (?:publications|proceedings)|ifmbe proceedings|"
    r"acm international conference proceeding series|ifip advances|advances in intelligent systems|"
    r"studies in |optics infobase|mechanisms and machine science|smart innovation|"
    r"ifac proceedings volumes|ifac-papersonline|aip conference proceedings|"
    r"journal of physics: conference series|procedia |energy procedia|springer proceedings)", re.I)

# Series whose edition names vary (host cities, renamed meetings, the conference's own workshops,
# the parts of a joint meeting), and conferences that share an acronym with another series:
# regex on the folded edition name (without a host conference, see own_name) -> (series key, display name).
SERIES_ALIASES = [
    (r"power ?tech\b", "POWERTECH", "IEEE PowerTech"),
    (r"engineering in medicine and biology", "EMBC",
     "IEEE Engineering in Medicine and Biology Conference (EMBC)"),
    (r"universities'? power engineering", "UPEC", "Universities Power Engineering Conference (UPEC)"),
    (r"global (?:tele)?communications conference|\bglobecom\b", "GLOBECOM",
     "IEEE Global Communications Conference (GLOBECOM)"),
    (r"vehicular technology conference", "VTC", "IEEE Vehicular Technology Conference (VTC)"),
    (r"international conference on communications?\b(?! technology| and)|\biccw?\b", "ICC",
     "IEEE International Conference on Communications (ICC)"),
    (r"international conference on image processing|\bicip\b", "ICIP",
     "IEEE International Conference on Image Processing (ICIP)"),
    (r"acoustics,? speech,? and signal processing|\bicassp\b", "ICASSP",
     "IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP)"),
    (r"intelligent robots and systems|\biros\b", "IROS",
     "IEEE/RSJ International Conference on Intelligent Robots and Systems (IROS)"),
    (r"international conference on robotics and automation|\bicra\b", "ICRA",
     "IEEE International Conference on Robotics and Automation (ICRA)"),
    (r"personal,? indoor,? and mobile radio|\bpimrc\b", "PIMRC",
     "IEEE International Symposium on Personal, Indoor and Mobile Radio Communications (PIMRC)"),
    (r"wireless communications and networking conference|\bwcnc\b", "WCNC",
     "IEEE Wireless Communications and Networking Conference (WCNC)"),
    (r"european (?:energy|electricity) market", "EEM",
     "International Conference on the European Energy Market (EEM)"),
    (r"conference on fuzzy systems|fuzz-ieee", "FUZZ-IEEE", "IEEE International Conference on Fuzzy Systems"),
    (r"european signal processing conference|\beusipco\b", "EUSIPCO",
     "European Signal Processing Conference (EUSIPCO)"),
    (r"european conference on antennas and propagation|\beucap\b", "EUCAP",
     "European Conference on Antennas and Propagation (EuCAP)"),
    (r"modern circuits and systems technologies|\bmocast\b", "MOCAST",
     "International Conference on Modern Circuits and Systems Technologies (MOCAST)"),
    (r"synergy ?med|energy transition in the mediterranean", "SYNERGY-MED",
     "International Conference on Energy Transition in the Mediterranean Area (SyNERGY MED)"),
    (r"computer[- ]based medical systems", "CBMS",
     "IEEE International Symposium on Computer-Based Medical Systems (CBMS)"),
    (r"conference on decision and control", "CDC", "IEEE Conference on Decision and Control (CDC)"),
    (r"midwest symposium on circuits and systems", "MWSCAS",
     "IEEE International Midwest Symposium on Circuits and Systems (MWSCAS)"),
    (r"international symposium (?:on|of) circuits and systems", "ISCAS",
     "IEEE International Symposium on Circuits and Systems (ISCAS)"),
    (r"international joint conference on neural networks", "IJCNN",
     "International Joint Conference on Neural Networks (IJCNN)"),
    (r"innovative smart grid technologies.{0,12}europe|isgt[ -]?europe", "ISGT-EUROPE",
     "IEEE PES Innovative Smart Grid Technologies Conference Europe (ISGT Europe)"),
    (r"black ?sea conference on communications", "BLACKSEACOM",
     "IEEE International Black Sea Conference on Communications and Networking (BlackSeaCom)"),
    (r"panhellenic conference on electronics and telecommunications", "PACET",
     "Panhellenic Conference on Electronics and Telecommunications (PACET)"),
    (r"electromagnetic field computation", "CEFC", "IEEE Conference on Electromagnetic Field Computation (CEFC)"),
    (r"artificial materials for novel wave|advanced electromagnetic materials in microwaves|"
     r"novel wave phenomena", "METAMATERIALS",
     "International Congress on Artificial Materials for Novel Wave Phenomena (Metamaterials)"),
    (r"\bintermag\b|international magnetics? conference", "INTERMAG",
     "IEEE International Magnetics Conference (INTERMAG)"),
    (r"emc europe", "EMC-EUROPE", "International Symposium on Electromagnetic Compatibility (EMC Europe)"),
    (r"environment and electrical engineering", "EEEIC",
     "International Conference on Environment and Electrical Engineering (EEEIC)"),
    (r"\bmedpower\b|mediterranean conference (?:and exhibition )?on power generation", "MEDPOWER",
     "Mediterranean Conference on Power Generation, Transmission, Distribution and Energy Conversion (MedPower)"),
    (r"mediterranean conference on control", "MED", "Mediterranean Conference on Control and Automation (MED)"),
    (r"european control conference", "ECC", "European Control Conference (ECC)"),
    (r"mediterranean electrotechnical conference", "MELECON",
     "IEEE Mediterranean Electrotechnical Conference (MELECON)"),
    (r"international conference on telecommunications and multimedia", "TEMU",
     "International Conference on Telecommunications and Multimedia (TEMU)"),
    (r"international conference on industrial technology", "ICIT",
     "IEEE International Conference on Industrial Technology (ICIT)"),
    (r"european wireless", "EW", "European Wireless Conference"),
    (r"international conference on pattern recognition\b", "ICPR",
     "International Conference on Pattern Recognition (ICPR)"),
    (r"progress in electromagnetics research symposium|photonics and electromagnetics research symposium",
     "PIERS", "Photonics and Electromagnetics Research Symposium (PIERS)"),
    (r"computers in cardiology|computing in cardiology", "CINC", "Computing in Cardiology (CinC)"),
    (r"electronics,? circuits,? and systems", "ICECS",
     "IEEE International Conference on Electronics, Circuits and Systems (ICECS)"),
    (r"multi-?media and expo", "ICME", "IEEE International Conference on Multimedia and Expo (ICME)"),
    (r"artificial intelligence applications and innovations", "AIAI",
     "Artificial Intelligence Applications and Innovations (AIAI)"),
    (r"mixed design of integrated circuits", "MIXDES",
     "International Conference on Mixed Design of Integrated Circuits and Systems (MIXDES)"),
    (r"ieee/wic\b", "WI-IAT", "IEEE/WIC/ACM Web Intelligence and Intelligent Agent Technology (WI-IAT)"),
    (r"universal access in human.computer interaction|hci international|\bhcii\b|"
     r"international conference on human.computer interaction(?! with mobile)", "HCII",
     "HCI International (incl. Universal Access in Human-Computer Interaction)"),
    (r"urs?i atlantic.{0,25}radio science", "AT-RASC", "URSI Atlantic Radio Science Meeting (AT-RASC)"),
    (r"general assembly and scientific symposium", "URSI-GASS", "URSI General Assembly and Scientific Symposium"),
    (r"medical informatics europe", "MIE", "Medical Informatics Europe (MIE)"),
    (r"european conference of the international federation for medical and biological", "EMBEC",
     "European Medical and Biological Engineering Conference (EMBEC)"),
    (r"rfid.?technolog(?:y|ies) and applications", "RFID-TA",
     "IEEE International Conference on RFID Technology and Applications (RFID-TA)"),
    (r"computational electromagnetics international workshop", "CEM-WORKSHOP",
     "Computational Electromagnetics International Workshop (CEM)"),
    (r"(?:evolving|evolutionary) fuzzy systems", "GEFS", "Genetic and Evolving Fuzzy Systems (GEFS)"),
    (r"\bflins\b|fuzzy logic and intelligent technologies in nuclear science", "FLINS",
     "International FLINS Conference (FLINS)"),
    (r"agents and data mining interaction", "ADMI", "Agents and Data Mining Interaction (ADMI)"),
    (r"mining software repositories", "MSR", "International Conference on Mining Software Repositories (MSR)"),
    # IQEC is held jointly with a CLEO conference; in the data only with CLEO/Europe (checked below)
    (r"lasers and ele?ctro-optics europe|cleo.europe|european quantum electronics|\beqec\b|\biqec\b", "CLEO-EUROPE",
     "Conference on Lasers and Electro-Optics Europe and European Quantum Electronics Conference (CLEO/Europe)"),
    # CLEO itself (after CLEO/Europe), with its QELS part ("CLEO: QELS_Fundamental Science")
    (r"conference on lasers and electro-?optics(?!\W*(?:europe|pacific))|^cleo\b(?!\W*(?:europe|pacific))|"
     r"\bqels\b|quantum electronics and laser science", "CLEO", "Conference on Lasers and Electro-Optics (CLEO)"),
    (r"audio engineering society convention|\bconvention (?:spring|fall) preprints\b", "AES-CONVENTION",
     "Audio Engineering Society Convention (AES)"),
    (r"antennas and propagation society international symposium|\bapsursi\b|\bap-s/|"
     r"ieee international symposium on antennas and propagation and", "AP-S",
     "IEEE International Symposium on Antennas and Propagation and USNC-URSI Radio Science Meeting (AP-S/URSI)"),
    # the 2005 symposium was held jointly with IEEE as the "IEEE/ACES International Conference on Wireless
    # Communications and Applied Computational Electromagnetics"
    (r"\baces(?:-monterey)? ?(?:19|20)\d{2}\b|review of progress in applied computational electromagnetics|"
     r"\bieee/aces\b", "ACES", "Applied Computational Electromagnetics Society Symposium (ACES)"),
    (r"(?:power and energy|power engineering) society general meeting|\bpesgm\b", "PESGM",
     "IEEE Power and Energy Society General Meeting (PESGM)"),
    (r"\bpcim asia\b", "PCIM-ASIA", "International Exhibition and Conference for Power Electronics, Intelligent "
     "Motion, Renewable Energy and Energy Management (PCIM Asia)"),
    (r"\birep\b|bulk power system dynamics and control", "IREP",
     "IREP Symposium on Bulk Power System Dynamics and Control (IREP)"),
    (r"conference on control applications", "CCA", "IEEE Conference on Control Applications (CCA)"),
    (r"optical fib(?:er|re) communications? conference|\bofc\b", "OFC", "Optical Fiber Communication Conference (OFC)"),
    (r"integrated photonics research", "IPR", "Integrated Photonics Research, Silicon and Nanophotonics (IPR)"),
    (r"annual (?:meeting|conference) of the ieee (?:photonics|lasers and electro-optics) society|"
     r"ieee photonics conference", "IPC", "IEEE Photonics Conference (IPC)"),
    (r"\binfocom\b", "INFOCOM", "IEEE Conference on Computer Communications (INFOCOM)"),
    (r"\broma international symposium on electromagnetic compatibility|\bemc\W*(?:\d+\W*)?roma\b", "EMC-ROMA",
     "International Symposium on Electromagnetic Compatibility, Rome (EMC Roma)"),
    (r"ieee (?:international )?symposium on electromagnetic compatibility", "EMC",
     "IEEE International Symposium on Electromagnetic Compatibility (EMC)"),
    (r"computer music conference", "ICMC", "International Computer Music Conference (ICMC)"),
    (r"systems,? man,? and cybernetics", "SMC", "IEEE International Conference on Systems, Man, and Cybernetics (SMC)"),
    (r"scientific conference on information technology", "ICIT-INFOTECH",
     "International Scientific Conference on Information Technology (ICIT)"),
]
SERIES_RX = [(re.compile(rx), key) for rx, key, _ in SERIES_ALIASES]
SERIES_NAMES = {key: name for _, key, name in SERIES_ALIASES}
# display names of frequent series identified by their acronym
SERIES_NAMES.update({
    "SEST": "International Conference on Smart Energy Systems and Technologies (SEST)",
    "ICLP": "International Conference on Lightning Protection (ICLP)",
    "ICHVE": "IEEE International Conference on High Voltage Engineering and Applications (ICHVE)",
    "MEDICON": "Mediterranean Conference on Medical and Biological Engineering and Computing (MEDICON)",
})
NOT_ACRONYMS = {"IEEE", "USA", "UK", "ACM", "IET", "IEE", "IFAC", "SPIE", "IFIP", "EAI", "PES", "RAS",
                "RSJ", "IAPR", "WIC", "URSI", "IMEKO", "CIGRE", "ITG", "VDE", "GMM", "ICST", "IAS", "AES",
                "SIAM", "INSTICC", "IEICE", "ASME", "AIP", "OSA", "CEUR", "LNCS", "CCIS", "IFMBE", "PART",
                "VOL", "AND"}
YEAR = r"(?:19[5-9]\d|20[0-3]\d)"
# acronym written next to a year: "EuCAP 2013", "UPEC2009", "ICIP'02", "EEM 12", "(ICIP 2000)"
ACR_YEAR_RX = re.compile(r"(?<![\w-])([A-Za-z][A-Za-z&+\-]*[A-Za-z])\s?(?:'\s?\d{2}|-?" + YEAR
                         + r"|\s\d{2}\b)(?!\d)")
TRAILING_ACR_RX = re.compile(r",\s*([A-Za-z][A-Za-z&+\-]*[A-Za-z])\s*$")   # "..., ICTON"
ORDINAL_WORDS = ("first second third fourth fifth sixth seventh eighth ninth tenth eleventh twelfth "
                 "thirteenth fourteenth fifteenth sixteenth seventeenth eighteenth nineteenth twentieth "
                 "thirtieth fortieth fiftieth").split()
ORDINAL_RX = re.compile(r"\b(?:(?:twenty|thirty|forty|fifty)[\s-]?)?(?:" + "|".join(ORDINAL_WORDS) + r")\b")


def _is_acronym(token: str) -> bool:
    return (len(token) >= 3 and token.upper() not in NOT_ACRONYMS
            and not re.fullmatch(r"[IVXLC]+", token) and sum(ch.isupper() for ch in token) >= 2)


def acronym(name: str) -> str | None:
    """Series acronym of an edition name ('EuCAP 2013', 'ICIP'02', '..., ICTON'), else None."""
    text = unicodedata.normalize("NFKD", name)
    found = [m.group(1).upper() for m in ACR_YEAR_RX.finditer(text) if _is_acronym(m.group(1))]
    if found:
        return found[-1]
    m = TRAILING_ACR_RX.search(text)
    if m and _is_acronym(m.group(1)) and m.group(1).isupper():
        return m.group(1)
    return None


def name_key(name: str) -> str:
    """Edition name without years, ordinals, edition numbers, 'Proceedings' and stop words."""
    t = fold(name)
    t = re.sub(r"\([^)]*\)|'\d{2}\b", " ", t)
    t = re.sub(YEAR, " ", t)
    t = re.sub(r"\b\d+\s*(?:st|nd|rd|th)\b", " ", t)
    t = ORDINAL_RX.sub(" ", t)
    t = re.sub(r"\b(?:i{1,3}|iv|vi{0,3}|ix|xi{0,3}|xiv|xv|xvi{0,3}|xix|xx)\b", " ", t)   # roman numerals
    t = re.sub(r"\b(?:proceedings?|proc|conference record|digest(?: of technical papers)?|annual|"
               r"biennial|edition|papers|vol\s*\d+|part\s*\d+)\b", " ", t)
    t = re.sub(r"[^a-z]+", " ", t.replace("&", " and "))
    t = re.sub(r"\b(?:the|of|on|in|and|for|ieee|international|conference|symposium|workshop|"
               r"congress|meeting|s|a)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip()


# A meeting held in conjunction with another one is a series of its own: the host conference named
# after the phrase is left out ("WALTD 2008 - Held in Conjunction with the 8th ... ICALT 2008" -> WALTD).
HOST_RX = re.compile(r"[\s,.;:–-]+(?:held\s+(?:jointly\s+)?with|(?:held\s+)?in\s+conjunction\s+with|"
                     r"co-?located\s+with|collocated\s+with|located\s+at)\b.*$", re.I | re.S)


def own_name(name: str) -> str:
    """Edition name without the conference it was held in conjunction with."""
    return HOST_RX.sub("", name).strip() or name


def series_key(name: str) -> str:
    """Curated alias, else the acronym, else the normalised name (of the meeting itself)."""
    own = own_name(name)
    low = fold(own)
    for rx, key in SERIES_RX:
        if rx.search(low):
            return key
    return acronym(own) or "name:" + name_key(own)


def assign_series(names: pd.Series) -> pd.Series:
    """Series key per edition name; a name-only key joins the acronym series whose edition
    names (with or without the acronym) normalise to the same text."""
    keys = names.map(series_key)
    nkeys = names.map(lambda name: name_key(own_name(name)))
    has_acr = ~keys.str.startswith("name:")
    lookup: dict[str, set[str]] = {}
    for nk, key in zip(nkeys[has_acr], keys[has_acr]):
        bare = " ".join(w for w in nk.split() if w not in key.lower().replace("-", " ").split())
        for variant in {nk, bare}:
            lookup.setdefault(variant, set()).add(key)
    joined = nkeys.map(lambda nk: next(iter(lookup[nk])) if len(lookup.get(nk, ())) == 1 else None)
    take = ~has_acr & joined.notna()
    keys[take] = joined[take]
    return keys


def edition_display(names: pd.Series, key: str) -> str:
    """Readable series name from its edition names (for series without a curated name)."""
    def clean(name: str) -> str:
        t = re.sub(r"^(?:proceedings(?: of)?(?: the)?|proc\.)\s*[-:]?\s*", "", name.strip(), flags=re.I)
        t = re.sub(r"^\s*(?:19|20)\d{2}\s+", "", t)
        t = ORDINAL_RX.sub("", re.sub(r"\b\d+\s*(?:st|nd|rd|th)\b\s*", "", t, flags=re.I))
        t = re.sub(r",?\s*[A-Za-z][\w&/+-]*['\s-]?(?:19|20)?\d{2}\s*(?:-\s*proceedings)?$", "", t)
        t = re.sub(r"\s*-\s*(?:conference )?proceedings$|,\s*proceedings$", "", t, flags=re.I)
        t = re.sub(r"\s+", " ", t).strip(" ,-:")
        return t[:1].upper() + t[1:]
    best = modal(names.map(clean))
    if key.startswith("name:") or re.search(rf"\b{re.escape(key)}\b", best, flags=re.I):
        return best
    return f"{best} ({key})"


edition = pubs["conference_name"].fillna(pubs["source_title"])
journal_keys = set(jkey[is_journal])
# A paper with no conference name takes its series from the source title only when the title names a
# meeting and is not a container; other titles are journals or generic series.
MEETING_RX = re.compile(r"\b(?:conference|symposium|congress|workshop|meeting|convention|colloquium)s?\b", re.I)
is_container = pubs["source_title"].str.match(CONTAINER_RX, na=False)
names_meeting = pubs["source_title"].str.contains(MEETING_RX, na=False)
no_series = is_conf & pubs["conference_name"].isna() & (is_container | ~names_meeting)
in_series = is_conf & ~no_series
assert (is_container | jkey.isin(journal_keys) | pubs["issn"].notna())[no_series].all(), \
    "notes: the conference papers without a series appeared in a journal or a generic series (serials, with an ISSN)"
series_of = assign_series(edition[in_series])
# Every edition of an aliased series matches its alias: another conference that only shares the
# acronym would otherwise join the series unnoticed (give it an alias of its own).
alias_rx = {key: rx for rx, key in SERIES_RX}
stray = [(key, name) for name, key in zip(edition[in_series], series_of)
         if key in alias_rx and not alias_rx[key].search(fold(own_name(name)))]
assert not stray, f"text: conferences that only share an acronym with an aliased series are kept apart: {stray[:3]}"
# a meeting held in conjunction with another conference does not take the host's acronym
hosted = [(key, name) for name, key in zip(edition[in_series], series_of)
          if own_name(name) != name and key == acronym(name) != acronym(own_name(name))]
assert not hosted, f"notes: a meeting held in conjunction with another conference is a series of its own: {hosted[:3]}"
# ... while a conference's own workshop proceedings count with it
icc_workshops = in_series & edition.str.contains(r"\bICC ?W(?:orkshops)?\b", na=False)
assert icc_workshops.any() and series_of[icc_workshops[in_series]].eq("ICC").all(), \
    "notes: the ICC Workshops count with ICC"
# IQEC counts with CLEO/Europe (alias above) only where the two were held as one meeting
edition_year = edition.str.extract(f"({YEAR})", expand=False).fillna(pubs["year"].astype(str))
is_iqec = in_series & edition.map(fold).str.contains(r"\biqec\b")
is_joint = is_iqec & edition.map(fold).str.contains(r"cleo.europe")
assert set(edition_year[is_iqec]) <= set(edition_year[is_joint]), \
    "aliases: an IQEC edition not held with CLEO/Europe (give it its own series)"
is_qels = in_series & edition.str.contains(r"\bQELS", na=False)
assert is_qels.any() and series_of[is_qels[in_series]].eq("CLEO").all(), "notes: QELS counts with CLEO"

# Venue of every record: journal, conference series, or the source title (books, containers).
pubs["venue"] = "S:" + jkey
pubs.loc[is_journal | (no_series & jkey.isin(journal_keys)), "venue"] = "J:" + jkey
pubs.loc[in_series, "venue"] = "C:" + series_of
pubs["venue_kind"] = np.select([is_journal, in_series], ["Journal", "Conference series"], "Other")
venue_names = pubs.sort_values("year", kind="stable").groupby("venue")["source_title"].last()  # latest title
for venue, names in edition[in_series].groupby(pubs.loc[in_series, "venue"]):
    key = venue[2:]
    venue_names[venue] = SERIES_NAMES.get(key) or edition_display(names.map(own_name), key)
pubs["venue_name"] = pubs["venue"].map(venue_names)
mdpi_venues = pubs.loc[is_mdpi & in_recent, "venue"].value_counts()      # MDPI papers of P_RECENT by venue
top_mdpi_journal, n_top_mdpi_journal = venue_names[mdpi_venues.index[0]], int(mdpi_venues.iloc[0])
assert mdpi_venues.iloc[0] > mdpi_venues.iloc[1], "text: one journal carried the most MDPI papers of P_RECENT"
venue_publisher = pubs.groupby("venue")["publisher_group"].agg(
    lambda s: modal(s[s.ne("Unknown")]) or "Unknown")      # main known publisher of each venue


def short_name(venue: str) -> str:
    """Acronym of a conference series named '... (ACRONYM)', else the full venue name."""
    name = venue_names[venue]
    m = re.search(r"\(([^()]{2,15})\)$", name) if venue.startswith("C:") else None
    return m.group(1) if m else name

n_venues = pubs["venue"].nunique()
n_journal_papers, n_conf_papers = int(is_journal.sum()), int(is_conf.sum())
n_book_papers = int(pubs["source_kind"].eq("book").sum())
assert n_journal_papers + n_conf_papers + n_book_papers == n_pubs, "text: the other records are books and chapters"
assert pubs.loc[is_conf, "doc_type"].eq("Conference paper").all(), \
    "text: the conference records are those Scopus types as conference papers"
n_journals = pubs.loc[is_journal, "venue"].nunique()
n_series = pubs.loc[in_series, "venue"].nunique()
n_series_papers, n_no_series = int(in_series.sum()), int(no_series.sum())

# ------------------------------------------------------------------------------ citation scores
# Normalised citation score (NCS) as on the Citation impact page: citations divided by the mean
# of the School's papers of the same year and type (articles and reviews together: too few
# reviews a year for an average of their own); articles, reviews and conference papers from
# NCS_FROM to NCS_TO, the papers with at least two full calendar years of citations (later papers
# have had too little time to be cited).
NCS_FROM, NCS_TO = 2000, PARTIAL - 3
CITABLE = ["Article", "Review", "Conference paper"]
in_ncs = pubs["year"].between(NCS_FROM, NCS_TO) & pubs["doc_type"].isin(CITABLE)
pubs["ncs_type"] = pubs["doc_type"].replace({"Review": "Article"})
pubs["ncs"] = metrics.normalized_citation_score(pubs[in_ncs], by=["year", "ncs_type"])
MIN_PAPERS = 10      # minimum number of scored papers for a venue's mean score


def above_average(scores: pd.Series) -> pd.Series:
    """Mean scores above the School average (1) at the precision shown on the page
    (two decimals): a venue shown as 1.00 counts as at the average, not above it."""
    return scores.round(2) > 1


def venue_table(df: pd.DataFrame) -> pd.DataFrame:
    """Papers, recent papers, active years, mean citations and mean NCS per venue."""
    g = df.groupby("venue")
    out = pd.DataFrame({
        "papers": g.size(),
        "recent": g["period"].agg(lambda s: int(s.eq(P_RECENT).sum())),
        "editions": g["year"].nunique(),
        "first": g["year"].min(),
        "last": g["year"].max(),
        "mean_cites": g["cited_by"].mean(),
        "n_scored": g["ncs"].count(),
        "ncs": g["ncs"].mean(),
    })
    out["name"] = out.index.map(venue_names)
    out["publisher"] = out.index.map(venue_publisher)
    out["ncs_shown"] = out["ncs"].where(out["n_scored"] >= MIN_PAPERS)
    out["years"] = out["first"].astype(str) + "–" + out["last"].astype(str)
    return out.sort_values(["papers", "name"], ascending=[False, True])


def top_with_ties(table: pd.DataFrame, n: int) -> tuple[pd.DataFrame, int]:
    """The n largest venues plus any venue tied with the n-th; also returns the cut-off."""
    cut = int(table["papers"].iloc[n - 1])
    return table[table["papers"] >= cut], cut


N_TOP_J, N_TOP_S = 20, 15       # rows of the journal and conference series tables (before ties)
journals = venue_table(pubs[is_journal])
top_journals, j_cut = top_with_ties(journals, N_TOP_J)
n_top_j = len(top_journals)
share_top_j = top_journals["papers"].sum() / n_journal_papers
n_top_j_ieee = int(top_journals["publisher"].eq("IEEE").sum())
top_j_name, top_j_papers = top_journals["name"].iloc[0], int(top_journals["papers"].iloc[0])
assert journals["papers"].iloc[0] > journals["papers"].iloc[1], "text: one journal has the most papers"
recent_j_rank = journals.sort_values(["recent", "papers"], ascending=False)
recent_j = recent_j_rank.iloc[0]
recent_j_name, recent_j_papers = recent_j["name"], int(recent_j["recent"])
assert recent_j_papers > recent_j_rank["recent"].iloc[1], "text: one journal is the most used of P_RECENT"
assert recent_j.name != journals.index[0], \
    "text: '<most-used journal> leads, but <another journal> has been the most-used journal of P_RECENT'"
n_top_j_scored = int(top_journals["ncs_shown"].notna().sum())
n_top_j_above = int(above_average(top_journals["ncs_shown"]).sum())
best_j = top_journals.loc[top_journals["ncs_shown"].idxmax()]
best_j_name, best_j_ncs, best_j_scored = best_j["name"], best_j["ncs_shown"], int(best_j["n_scored"])
best_j_scores = pubs.loc[pubs["venue"].eq(best_j.name) & is_journal, "ncs"].dropna()
best_j_top_share = best_j_scores.max() / best_j_scores.sum()   # part of the score sum from its best paper
best_j_median = best_j_scores.median()
top_j_ncs = top_journals["ncs_shown"].iloc[0]
assert pd.notna(top_j_ncs), "text: the most-used journal has a normalised score"
# open-access journals with a broad scope: IEEE Access and the MDPI journals
is_broad_oa = is_journal & (pubs["venue"].eq("J:ieee access") | is_mdpi)
assert pubs.loc[is_broad_oa, "oa_status"].eq("gold").all(), "text: these journals are fully open access (gold)"
broad_prev, broad_recent = (is_broad_oa[is_journal & m].mean() for m in (in_prev, in_recent))

series = venue_table(pubs[in_series])
top_series, s_cut = top_with_ties(series, N_TOP_S)
n_top_s = len(top_series)
share_top_s = top_series["papers"].sum() / n_conf_papers
n_top_s_ieee = int(top_series["publisher"].eq("IEEE").sum())
top_s_name, top_s_papers = top_series["name"].iloc[0], int(top_series["papers"].iloc[0])
top_s_editions = int(top_series["editions"].iloc[0])
assert series["papers"].iloc[0] > series["papers"].iloc[1], "text: one conference series has the most papers"
recent_s_rank = series.sort_values(["recent", "papers"], ascending=False)
recent_s = recent_s_rank.iloc[0]
recent_s_name, recent_s_papers = recent_s["name"], int(recent_s["recent"])
assert recent_s_papers > recent_s_rank["recent"].iloc[1], "text: one conference series is the most used of P_RECENT"
# topic of the leading series (hand-assigned from their names)
POWER_SERIES = {"UPEC", "POWERTECH", "MEDPOWER", "EEM", "EEEIC", "SEST", "SYNERGY-MED", "ISGT-EUROPE",
                "ICHVE", "ICLP", "SIPDA"}
COMM_SERIES = {"ICC", "GLOBECOM", "WCNC", "PIMRC", "VTC", "EUCAP", "WPMC"}
OTHER_SERIES = {"EMBC", "MOCAST", "MED", "METAMATERIALS", "ECC", "ICIP", "ICRA", "IROS", "FUZZ-IEEE"}
top_keys = [v[2:] for v in top_series.index]
assert set(top_keys) <= POWER_SERIES | COMM_SERIES | OTHER_SERIES, \
    "text: every leading series has a hand-assigned topic group (assign the new ones above)"
power_top = [k for k in top_keys if k in POWER_SERIES]
comm_top = [k for k in top_keys if k in COMM_SERIES]
n_power_top, n_comm_top = len(power_top), len(comm_top)
assert n_power_top >= 2 and n_comm_top >= 2, "text: several power and several communications series lead"
power_top_papers = int(top_series.loc[["C:" + k for k in power_top], "papers"].sum())
power_top_text = join_and([short_name("C:" + k) for k in power_top])
comm_top_text = join_and([short_name("C:" + k) for k in comm_top])
# Names for the series table: the acronym first and frequent words shortened, so that the table
# fits the text column (the full names are in the venue figure's hover).
SHORT_WORDS = [(r"\bInternational\b", "Int."), (r"\bConference\b", "Conf."), (r"\bSymposium\b", "Symp."),
               (r"\bEngineering\b", "Eng.")]


def table_name(venue: str) -> str:
    """'Universities Power Engineering Conference (UPEC)' -> 'UPEC – Universities Power Eng. Conf.'"""
    name = venue_names[venue]
    m = re.search(r"\s*\(([^()]{2,15})\)$", name)
    if m:
        name = f"{m.group(1)} – {name[:m.start()]}"
    for rx, short in SHORT_WORDS:
        name = re.sub(rx, short, name)
    return name


top_series = top_series.assign(short=[table_name(v) for v in top_series.index])
best_s = top_series.loc[top_series["ncs_shown"].idxmax()]
best_s_name, best_s_ncs = best_s["name"], best_s["ncs_shown"]

# ------------------------------------------------------------------------------ venue portfolio
scored = pubs[in_ncs & pubs["venue_kind"].ne("Other")]
gs = scored.groupby("venue")
portfolio = pd.DataFrame({
    "kind": gs["venue_kind"].first(),
    "papers": gs.size(),
    "cites": gs["cited_by"].sum(),
    "mean_cites": gs["cited_by"].mean(),
    "ncs": gs["ncs"].mean(),
})
portfolio = portfolio[portfolio["papers"] >= MIN_PAPERS].sort_values("papers", ascending=False)
portfolio["name"] = portfolio.index.map(venue_names)
portfolio["publisher"] = portfolio.index.map(venue_publisher)
n_port = len(portfolio)
n_port_j, n_port_c = (int(portfolio["kind"].eq(k).sum()) for k in ("Journal", "Conference series"))
n_scored = int(in_ncs.sum())
share_port = portfolio["papers"].sum() / n_scored
n_port_above = int(above_average(portfolio["ncs"]).sum())
port_j_above, port_c_above = (
    above_average(portfolio.loc[portfolio["kind"].eq(k), "ncs"]).mean() for k in ("Journal", "Conference series"))
rho_port = portfolio["papers"].rank().corr(portfolio["ncs"].rank())      # Spearman rank correlation
rho_text = ("no clear relation" if abs(rho_port) < 0.2 else
            "a weak relation" if abs(rho_port) < 0.4 else "a moderate relation")
assert abs(rho_port) < 0.4, "key findings: how often a venue is used 'says little' about its citations"
port_top = portfolio.iloc[0]                                # the venue with the most scored papers
assert portfolio["papers"].iloc[0] > portfolio["papers"].iloc[1], "text: one venue has the most scored papers"
port_top_name, port_top_papers, port_top_ncs = port_top["name"], int(port_top["papers"]), port_top["ncs"]
port_best = portfolio.loc[portfolio["ncs"].idxmax()]
port_best_name, port_best_ncs, port_best_papers = port_best["name"], port_best["ncs"], int(port_best["papers"])
SCORED_UNIT = {"Journal": "articles and reviews", "Conference series": "conference papers"}
port_top_unit, port_best_unit = SCORED_UNIT[port_top["kind"]], SCORED_UNIT[port_best["kind"]]
n_port_ieee = int(portfolio["publisher"].eq("IEEE").sum())

# ------------------------------------------------------------------------------ venue diversity
years = pd.RangeIndex(FIRST_YEAR, PARTIAL + 1, name="year")
yearly = (pubs.groupby("year").agg(papers=("eid", "size"), venues=("venue", "nunique"))
          .reindex(years, fill_value=0))
full_years = yearly.loc[:LAST_FULL]
venues_peak_year = int(full_years["venues"].idxmax())
venues_peak, papers_peak_year = (int(full_years.at[venues_peak_year, c]) for c in ("venues", "papers"))
venues_last, papers_last = (int(full_years.at[LAST_FULL, c]) for c in ("venues", "papers"))
up_to_peak = full_years.loc[:venues_peak_year]
assert up_to_peak["venues"].corr(up_to_peak["papers"]) > 0.9, "text: the venues grew with output until their peak"
assert venues_peak_year < LAST_FULL and papers_last > papers_peak_year and venues_last < venues_peak, \
    "text: the venues peaked before the latest full year; output has grown since, the venues have not"


def rarefied_venues(counts: np.ndarray, n: int) -> float:
    """Expected number of distinct venues among n papers drawn at random (without
    replacement) from papers with the given counts per venue (rarefaction, Hurlbert 1971)."""
    total = int(counts.sum())
    if n >= total:
        return float(len(counts))
    log_all = math.lgamma(total + 1) - math.lgamma(n + 1) - math.lgamma(total - n + 1)
    missed = sum(math.exp(math.lgamma(total - c + 1) - math.lgamma(n + 1) - math.lgamma(total - c - n + 1)
                          - log_all) for c in counts if total - c >= n)
    return float(len(counts) - missed)


DIV_PERIODS = BIG_PERIODS                                 # from P_FIRST on


def period_leaders(counts: pd.Series, n: int = 3) -> list[tuple[str, int]]:
    """The n most-used venues of a period and every venue tied with the n-th (by papers, then name)."""
    top = counts[counts >= counts.iloc[n - 1]]
    return sorted(((venue_names[v], int(c)) for v, c in top.items()), key=lambda vc: (-vc[1], vc[0]))


assert per_n[DIV_PERIODS].is_monotonic_increasing, "text: output grew from period to period"
period_counts = {p: pubs.loc[pubs["period"].eq(p), "venue"].value_counts() for p in DIV_PERIODS}
# draw size: the smallest period from P_BASE on, rounded down to hundreds
RAREFY_N = int(per_n[PERIODS[PERIODS.index(P_BASE):]].min()) // 100 * 100
RARE_MISSING = [p for p in DIV_PERIODS if per_n[p] < RAREFY_N]      # too small for such a draw
rare_note = "rarefaction" + (f"; {join_and(RARE_MISSING)} {'has' if len(RARE_MISSING) == 1 else 'have'} "
                             "fewer publications" if RARE_MISSING else "")
div = pd.DataFrame({
    "papers": [int(period_counts[p].sum()) for p in DIV_PERIODS],
    "venues": [len(period_counts[p]) for p in DIV_PERIODS],
    "top10": [metrics.top_share(period_counts[p].to_numpy(), n=10) for p in DIV_PERIODS],
    "rarefied": [rarefied_venues(period_counts[p].to_numpy(), RAREFY_N) if per_n[p] >= RAREFY_N else np.nan
                 for p in DIV_PERIODS],
    "leaders": [period_leaders(period_counts[p]) for p in DIV_PERIODS],
}, index=pd.Index(DIV_PERIODS, name="period"))
top10_first, top10_prev, top10_recent = (div.at[p, "top10"] for p in (P_FIRST, P_PREV, P_RECENT))
i_stable = DIV_PERIODS.index(P_STABLE)
top10_lo, top10_hi = div["top10"].iloc[i_stable:].agg(["min", "max"])
assert div["top10"].iloc[:i_stable].min() > top10_hi, \
    "text: the top-10 share fell from the first period and has stayed in a band since P_STABLE"
venues_prev, venues_recent = (int(div.at[p, "venues"]) for p in (P_PREV, P_RECENT))
papers_prev, papers_recent = (int(div.at[p, "papers"]) for p in (P_PREV, P_RECENT))
rare_base, rare_prev, rare_recent = (div.at[p, "rarefied"] for p in (P_BASE, P_PREV, P_RECENT))
venues_recent_rarefied = rarefied_venues(period_counts[P_RECENT].to_numpy(), papers_prev)
assert rare_base < rare_prev > rare_recent, \
    "text: the rarefied venue count rose from P_BASE to P_PREV and fell in P_RECENT"
assert papers_recent > papers_prev and venues_recent < venues_prev and venues_recent_rarefied < venues_prev, \
    "text: P_RECENT has more papers than P_PREV in fewer venues, also when drawn down to the size of P_PREV"
recent_counts = period_counts[P_RECENT]
assert recent_counts.iloc[0] > recent_counts.iloc[1] and recent_counts.iloc[2] > recent_counts.iloc[3], \
    "text: the venue that led P_RECENT and its three most-used venues are not tied"
recent_leaders = join_and([f"{short_name(v)} ({n}{' papers' if i == 0 else ''})"
                           for i, (v, n) in enumerate(period_counts[P_RECENT].iloc[:3].items())])
lead_recent_name = short_name(period_counts[P_RECENT].index[0])     # most-used venue of any kind
lead_recent_papers = int(period_counts[P_RECENT].iloc[0])

Where does the School publish its research? This page groups the {eval}`dp.fmt_int(n_pubs)` Scopus
records from {eval}`str(FIRST_YEAR)` to the export of {eval}`AS_OF_TEXT` by publisher, journal and
conference series. It follows how the publisher mix has changed, lists the journals and conference series
used most, compares how the papers in frequently used venues are cited and asks whether output is spread
over more or fewer venues. Journal papers are the {eval}`dp.fmt_int(n_journal_papers)` records of journal
document types (articles, reviews, letters, editorials and similar items), conference papers the
{eval}`dp.fmt_int(n_conf_papers)` records that Scopus types as conference papers; the other
{eval}`dp.fmt_int(n_book_papers)` records are books and book chapters. Publisher and conference names were
normalised as described under [Notes & caveats](#venues-notes). {eval}`str(PARTIAL)` is a partial year: it
is included in totals and tables and marked in the yearly chart, but left out of the period comparisons.

In [ ]:
dp.kpi_html([
    dp.KPI("Publication venues", n_venues, "journals, conference series, books and other sources"),
    dp.KPI("Journals", n_journals, f"{dp.fmt_int(n_journal_papers)} journal papers"),
    dp.KPI("Conference series", n_series,
           f"{dp.fmt_int(n_series_papers)} of {dp.fmt_int(n_conf_papers)} conference papers"),
    dp.KPI("Published by IEEE", dp.fmt_pct(share_ieee), f"{dp.fmt_pct(ieee_recent)} in {P_RECENT}"),
], min_width=140)

:::{important} Key findings
- IEEE published {eval}`dp.fmt_pct(share_ieee)` of all records ({eval}`dp.fmt_int(n_ieee)`). Its share
  rose from {eval}`dp.fmt_pct(ieee_first)` in {eval}`P_FIRST` to {eval}`dp.fmt_pct(ieee_prev)` in
  {eval}`P_PREV` and was {eval}`dp.fmt_pct(ieee_recent)` in {eval}`P_RECENT`, when IEEE published
  {eval}`dp.fmt_pct(ieee_conf_recent)` of the conference papers and {eval}`dp.fmt_pct(ieee_jour_recent)` of the
  journal papers.
- MDPI, first used in {eval}`str(mdpi_first)`, grew from {eval}`dp.fmt_pct(mdpi_prev)` of output in
  {eval}`P_PREV` to {eval}`dp.fmt_pct(mdpi_recent)` in {eval}`P_RECENT` ({eval}`dp.fmt_int(n_mdpi_recent)`
  papers, {eval}`dp.fmt_pct(mdpi_jour_recent)` of the journal papers). Its annual share
  {eval}`mdpi_annual_text`.
- The most-used journal is {eval}`top_j_name`, with {eval}`dp.fmt_int(top_j_papers)` papers, and the
  most-used conference series the {eval}`top_s_name`, with {eval}`dp.fmt_int(top_s_papers)`. In
  {eval}`P_RECENT` {eval}`lead_recent_name` led all venues with {eval}`dp.fmt_int(lead_recent_papers)` papers.
- How often the School uses a venue says little about how its papers there are cited: of the
  {eval}`str(n_port)` journals and conference series with at least {eval}`str(MIN_PAPERS)` articles, reviews
  or conference papers in {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`, {eval}`str(n_port_above)` are cited above
  the School average for papers of the same year and type, and the rank correlation between a venue's number of papers and its
  mean normalised score is {eval}`dp.fmt_num(rho_port, 2)`.
- Recent output is concentrated in fewer venues: the {eval}`dp.fmt_int(papers_recent)` papers of
  {eval}`P_RECENT` appeared in {eval}`dp.fmt_int(venues_recent)` venues, against
  {eval}`dp.fmt_int(venues_prev)` for the {eval}`dp.fmt_int(papers_prev)` papers of {eval}`P_PREV`. At that
  smaller size (a random draw of {eval}`dp.fmt_int(papers_prev)` papers), the {eval}`P_RECENT` output would
  cover about {eval}`dp.fmt_int(venues_recent_rarefied)` venues.
:::

In [ ]:
#| label: cell-venues-publishers
ticks = [f"{p}<br>n = {dp.fmt_int(per_n[p])}" for p in PERIODS]
# each segment starts where the previous groups end (explicit base: empty segments are left out
# of the bars and the hover without shifting the segments above them)
bases = per_share.cumsum(axis=1).shift(axis=1, fill_value=0)
fig = go.Figure()
for name in PUBLISHER_ORDER:
    marker = dict(color=PUBLISHER_COLORS[name], opacity=1)
    if name == "Unknown":   # outlined and hatched: no publisher could be determined
        marker = dict(color="rgba(0,0,0,0)", line=dict(color=dp.OTHER_COLOR, width=1),
                      pattern=dict(shape="x", fgcolor=dp.OTHER_COLOR, size=6, solidity=0.3))
    fig.add_bar(x=ticks, y=per_share[name].where(per_counts[name] > 0), base=bases[name], name=name,
                marker=marker, customdata=np.column_stack([per_share[name], per_counts[name]]),
                hovertemplate="%{customdata[0]:.1%} (%{customdata[1]:,} papers)")
fig.update_layout(barmode="overlay", hovermode="x unified", margin=dict(t=24))
fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="Share of publications")
fig.show()

## Publishers

Scopus leaves the Publisher field empty for {eval}`dp.fmt_pct(share_pub_missing)` of the records
({eval}`dp.fmt_int(n_pub_missing)`), most of them from {eval}`GAP_TEXT`. The gaps were filled from the DOI
prefix ({eval}`dp.fmt_int(n_by_doi)` records), the venue name ({eval}`dp.fmt_int(n_by_name)`) and other records
of the same journal ({eval}`dp.fmt_int(n_by_journal)`), which leaves {eval}`dp.fmt_int(n_unknown)` records
({eval}`dp.fmt_pct(share_unknown)`) with an unknown publisher; publishers are grouped under their current
owner (see the [notes](#venues-notes)).

:::{figure} #cell-venues-publishers
:label: fig-venues-publishers
Share of publications by publisher group and five-year period; the number of publications of each period
is given under its bar. Other: all publishers outside the six largest groups. Unknown: no publisher could
be determined. {eval}`str(PARTIAL)` is partial and not shown.
:::

IEEE has published more than half of the School's output in every period since
{eval}`ieee_major_from` ([](#fig-venues-publishers)): its share grew from {eval}`dp.fmt_pct(ieee_first)` in
{eval}`P_FIRST` to {eval}`dp.fmt_pct(ieee_prev)` in {eval}`P_PREV` and was {eval}`dp.fmt_pct(ieee_recent)` in
{eval}`P_RECENT`, with the largest lead in conferences ({eval}`dp.fmt_pct(ieee_conf_recent)` of the conference
papers of {eval}`P_RECENT`, against {eval}`dp.fmt_pct(ieee_jour_recent)` of the journal papers). Elsevier's
share fell from {eval}`dp.fmt_pct(els_first)` in {eval}`P_FIRST` to {eval}`dp.fmt_pct(els_base)` in
{eval}`P_BASE` and {eval}`dp.fmt_pct(els_recent)` in {eval}`P_RECENT`, while Springer Nature has stayed between
{eval}`dp.fmt_pct(spr_lo)` and {eval}`dp.fmt_pct(spr_hi)` since {eval}`P_FIRST`. The fastest-growing of the six
largest groups is MDPI: first used in {eval}`str(mdpi_first)`, it carried {eval}`dp.fmt_int(n_mdpi_recent)`
papers in {eval}`P_RECENT` ({eval}`dp.fmt_pct(mdpi_recent)` of output, up from {eval}`dp.fmt_pct(mdpi_prev)`
in {eval}`P_PREV`), most often in {eval}`top_mdpi_journal` ({eval}`dp.fmt_int(n_top_mdpi_journal)`), and its
annual share {eval}`mdpi_annual_text`. Unknown publishers reach at most {eval}`dp.fmt_pct(unk_max)` of a
period's output ({eval}`unk_max_period`), too few to change these trends.

In [ ]:
#| label: cell-venues-journals
dp.table_html(
    top_journals,
    columns={"name": "Journal", "publisher": "Publisher", "papers": "Papers", "recent": RECENT_COL,
             "years": "Years", "mean_cites": "Cites/paper", "ncs_shown": "NCS"},
    formats={"mean_cites": lambda v: dp.fmt_num(v, 1), "ncs_shown": lambda v: dp.fmt_num(v, 2)},
    numeric=["papers", "recent", "years", "mean_cites", "ncs_shown"],
)

## Journals

:::{figure} #cell-venues-journals
:kind: table
:label: tbl-venues-journals
The {eval}`str(n_top_j)` journals with the most School papers (at least {eval}`str(j_cut)`: the
{eval}`str(N_TOP_J)` largest, with ties). Papers: journal papers of all years to {eval}`AS_OF_TEXT`;
{eval}`RECENT_COL`: papers published {eval}`P_RECENT`; Years: first and last year with a paper. Cites/paper:
mean Scopus citations per paper at the export date. NCS: mean normalised citation score of the journal's
articles and reviews of {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)` (1 = the School average for articles and
reviews of the same year), shown for journals with at least {eval}`str(MIN_PAPERS)` such papers.
:::

The School's {eval}`dp.fmt_int(n_journal_papers)` journal papers appeared in
{eval}`dp.fmt_int(n_journals)` journals. The {eval}`str(n_top_j)` in [](#tbl-venues-journals) carry
{eval}`dp.fmt_pct(share_top_j)` of them, and {eval}`str(n_top_j_ieee)` of these are IEEE journals.
{eval}`top_j_name` leads with {eval}`dp.fmt_int(top_j_papers)` papers, but {eval}`recent_j_name` has been the
most-used journal of {eval}`P_RECENT` ({eval}`dp.fmt_int(recent_j_papers)` papers). Together, IEEE Access and
the MDPI journals, all of them fully open access, carried {eval}`dp.fmt_pct(broad_recent)` of the journal
papers of {eval}`P_RECENT`, against {eval}`dp.fmt_pct(broad_prev)` in {eval}`P_PREV`.

Citations per paper are not comparable between journals used in different decades, since older papers have
had longer to be cited; the normalised score removes this age effect. Of the {eval}`str(n_top_j_scored)`
journals with a score, {eval}`str(n_top_j_above)` are above the School average. The highest score,
{eval}`dp.fmt_num(best_j_ncs, 2)` for {eval}`best_j_name`, rests on {eval}`str(best_j_scored)` papers, one of
which contributes {eval}`dp.fmt_pct(best_j_top_share)` of the sum of their scores (median score
{eval}`dp.fmt_num(best_j_median, 2)`). The most-used journal, {eval}`top_j_name`, scores
{eval}`dp.fmt_num(top_j_ncs, 2)`.

In [ ]:
#| label: cell-venues-series
dp.table_html(
    top_series,
    columns={"short": "Conference series", "papers": "Papers", "editions": "Editions", "recent": RECENT_COL,
             "years": "Years", "mean_cites": "Cites/paper", "ncs_shown": "NCS"},
    formats={"mean_cites": lambda v: dp.fmt_num(v, 1), "ncs_shown": lambda v: dp.fmt_num(v, 2)},
    numeric=["papers", "editions", "recent", "years", "mean_cites", "ncs_shown"],
)

## Conference series

:::{figure} #cell-venues-series
:kind: table
:label: tbl-venues-series
The {eval}`str(n_top_s)` conference series with the most School papers (at least {eval}`str(s_cut)`:
the {eval}`str(N_TOP_S)` largest, with ties), named by their acronym (Int.: International, Conf.: Conference,
Symp.: Symposium, Eng.: Engineering); IEEE is the main publisher of {eval}`str(n_top_s_ieee)` of them.
Editions: years with at least one School paper. NCS compares each
conference paper of {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)` with the School's conference papers of the
same year (shown for at least {eval}`str(MIN_PAPERS)` such papers); other columns as in
[](#tbl-venues-journals).
:::

Of the {eval}`dp.fmt_int(n_conf_papers)` conference papers, {eval}`dp.fmt_int(n_series_papers)` could be
assigned to one of {eval}`dp.fmt_int(n_series)` conference series, identified from the Scopus conference name
(or, where that is missing, the source title) by the conference acronym or by the name without years and
edition numbers; editions of one series that are named differently, for example after their host city, are
joined through a list of aliases. The {eval}`str(n_top_s)` series
in [](#tbl-venues-series) hold {eval}`dp.fmt_pct(share_top_s)` of all conference papers, and IEEE is the
main publisher of {eval}`str(n_top_s_ieee)` of them. The {eval}`top_s_name` leads with
{eval}`dp.fmt_int(top_s_papers)` papers in {eval}`str(top_s_editions)` editions; in {eval}`P_RECENT` the
most-used series was the {eval}`recent_s_name`, with {eval}`dp.fmt_int(recent_s_papers)` papers.
Among them are {eval}`str(n_power_top)` power and energy meetings ({eval}`power_top_text`; together
{eval}`dp.fmt_int(power_top_papers)` papers) and {eval}`str(n_comm_top)` communications and antennas
conferences ({eval}`comm_top_text`). Among the {eval}`str(n_top_s)`, the {eval}`best_s_name` has the
highest normalised score ({eval}`dp.fmt_num(best_s_ncs, 2)`).

In [ ]:
#| label: cell-venues-portfolio
COLOURED = TOP_PUBLISHERS              # the six groups of the publisher figure, in its colours; the rest in grey
pf = portfolio.assign(
    hue=portfolio["publisher"].where(portfolio["publisher"].isin(COLOURED), "Other"),
    label=portfolio["name"].map(lambda s: dp.wrap_label(s, 48, 3)),
)
MAX_DIAMETER = 44                      # px, for the venue with the most citations
size_ref = 2.0 * pf["cites"].max() / MAX_DIAMETER ** 2
fig = go.Figure()
for hue in COLOURED + ["Other"]:
    shown = False
    for kind, symbol in (("Journal", "circle"), ("Conference series", "diamond")):
        d = pf[pf["hue"].eq(hue) & pf["kind"].eq(kind)]
        if d.empty:
            continue
        fig.add_scatter(
            x=d["papers"], y=d["ncs"], mode="markers",
            name=hue if hue != "Other" else "Other publishers", legendgroup=hue, showlegend=not shown,
            marker=dict(size=d["cites"], sizemode="area", sizeref=size_ref, sizemin=5, symbol=symbol,
                        color=PUBLISHER_COLORS[hue], opacity=0.75),
            customdata=d[["label", "kind", "publisher", "cites", "mean_cites"]].to_numpy(dtype=object),
            hovertemplate=("<b>%{customdata[0]}</b><br>%{customdata[1]}, %{customdata[2]}"
                           f"<br>Publications {NCS_FROM}–{NCS_TO}: " "%{x}<br>Mean normalised score: %{y:.2f}"
                           "<br>Citations: %{customdata[3]:,} (%{customdata[4]:.1f} per paper)<extra></extra>"),
        )
        shown = True
# direct labels: the most-used and the highest-scoring venue
labelled = pf.loc[list(dict.fromkeys([pf["papers"].idxmax(), pf["ncs"].idxmax()]))]
for _, row in labelled.iterrows():
    radius = MAX_DIAMETER / 2 * (row["cites"] / pf["cites"].max()) ** 0.5
    left = row["papers"] >= pf["papers"].quantile(0.9)     # near the right edge: label to the left
    fig.add_annotation(x=np.log10(row["papers"]), y=row["ncs"], text=dp.wrap_label(row["name"], 30, 2),
                       xanchor="right" if left else "left", align="right" if left else "left",
                       xshift=-(radius + 4) if left else radius + 4, showarrow=False)
fig.add_hline(y=1, line=dict(color=dp.INK["muted"], width=1, dash="dot"), layer="below",
              annotation_text="School average", annotation_position="top right")
fig.update_xaxes(type="log", tickvals=[10, 20, 30, 50, 80, 150, 300], showgrid=True,
                 title_text=f"Publications {NCS_FROM}–{NCS_TO} (log scale)")
fig.update_yaxes(rangemode="tozero", title_text="Mean normalised citation score")
fig.update_layout(height=480, margin=dict(t=24))
fig.show()

## Citation impact by venue

:::{figure} #cell-venues-portfolio
:label: fig-venues-portfolio
Journals (circles) and conference series (diamonds) with at least {eval}`str(MIN_PAPERS)` School
articles and reviews (journals) or conference papers (series) published
{eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`: the number of these papers (log scale) against their mean
normalised citation score. Bubble area: citations of these papers. Colour: the venue's main publisher, as in
[](#fig-venues-publishers) (grey: other publishers). The dotted line marks the School average (1).
:::

In all, {eval}`str(n_port)` venues ({eval}`str(n_port_j)` journals and {eval}`str(n_port_c)` conference
series) had at least {eval}`str(MIN_PAPERS)` such papers; together they account for
{eval}`dp.fmt_pct(share_port)` of the {eval}`dp.fmt_int(n_scored)` articles, reviews and conference papers of
the period ([](#fig-venues-portfolio)). Of these, {eval}`str(n_port_above)` are cited above the School
average for papers of the same year and type: {eval}`dp.fmt_pct(port_j_above)` of the journals and
{eval}`dp.fmt_pct(port_c_above)` of the conference series, each compared with papers of its own type. There
is {eval}`rho_text` between how often the School used a venue and how well its papers there are cited
(Spearman rank correlation {eval}`dp.fmt_num(rho_port, 2)`): the most-used venue, {eval}`port_top_name`
({eval}`dp.fmt_int(port_top_papers)` {eval}`port_top_unit` of {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`),
scores {eval}`dp.fmt_num(port_top_ncs, 2)`, while the highest score, {eval}`dp.fmt_num(port_best_ncs, 2)` for
{eval}`port_best_name`, comes from {eval}`dp.fmt_int(port_best_papers)` {eval}`port_best_unit`. Venue averages are associations: they mix the reach of the venue
with the topics, teams and collaborations of the papers sent to it, and a single highly cited paper can lift
the mean of a venue with few papers.

In [ ]:
#| label: cell-venues-diversity
counts = (yearly.rename(columns={"papers": "Publications", "venues": "Distinct venues"})
          .reset_index().melt(id_vars="year", var_name="series", value_name="n"))
fig = dp.year_line(counts, "year", "n", color="series",
                   colors={"Publications": dp.INK["muted"], "Distinct venues": dp.CATEGORICAL[0]},
                   partial_year=PARTIAL, as_of=AS_OF, value_title="Number per year")
fig.show()

## Venue diversity

:::{figure} #cell-venues-diversity
:label: fig-venues-diversity
Publications and distinct venues (journals, conference series, books and other sources) per year.
{eval}`str(PARTIAL)` is partial (records to {eval}`AS_OF.strftime('%d %B').lstrip('0')`).
:::

The number of distinct venues grew with output until {eval}`str(venues_peak_year)`, when the School's
{eval}`dp.fmt_int(papers_peak_year)` papers appeared in {eval}`dp.fmt_int(venues_peak)` venues. Output has
grown since, but the number of venues has not: the {eval}`dp.fmt_int(papers_last)` papers of
{eval}`str(LAST_FULL)` appeared in {eval}`dp.fmt_int(venues_last)` venues ([](#fig-venues-diversity)).

In [ ]:
#| label: cell-venues-top10
from plotly.subplots import make_subplots

leaders = ["<br>".join(f"{dp.wrap_label(name, 60, 1)} ({n})" for name, n in row) for row in div["leaders"]]
custom = np.column_stack([div["papers"].map(dp.fmt_int), div["venues"].map(dp.fmt_int), leaders])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.14,
                    subplot_titles=["Share of the period's publications in its ten most-used venues",
                                    f"Expected distinct venues among {RAREFY_N} random publications"])
fig.add_bar(x=div.index, y=div["top10"], row=1, col=1, marker_color=dp.CATEGORICAL[0],
            texttemplate="%{y:.1%}", textposition="outside", customdata=custom,
            hovertemplate=("<b>%{x}</b><br>Ten most-used venues: %{y:.1%} of %{customdata[0]} publications"
                           "<br>Most used:<br>%{customdata[2]}<extra></extra>"))
fig.add_bar(x=div.index, y=div["rarefied"], row=2, col=1, marker_color=dp.CATEGORICAL[0],
            texttemplate="%{y:.0f}", textposition="outside", customdata=custom,
            hovertemplate=(f"<b>%{{x}}</b><br>Expected among {RAREFY_N} random publications: "
                           "%{y:.0f} venues<br>Observed: %{customdata[1]} venues for %{customdata[0]} "
                           "publications<extra></extra>"))
fig.update_yaxes(tickformat=".0%", range=[0, div["top10"].max() * 1.25], title_text="Share", row=1, col=1)
fig.update_yaxes(range=[0, div["rarefied"].max() * 1.18], title_text="Venues", row=2, col=1)
fig.update_annotations(font=dict(size=13), x=0, xanchor="left")
fig.update_layout(height=540, showlegend=False, margin=dict(t=36))
fig.show()

:::{figure} #cell-venues-top10
:label: fig-venues-top10
Concentration and diversity of venues by five-year period, from {eval}`P_FIRST` (earlier periods
have fewer than {eval}`str(MIN_PERIOD_N)` publications). Top: share of the period's publications that appeared
in the ten venues used most in that period. Bottom: expected number of distinct venues among
{eval}`str(RAREFY_N)` publications drawn at random from the period ({eval}`rare_note`).
The hover lists the three most-used venues of each period, with any venue tied for third place.
:::

As output grew, the share of the ten most-used venues fell from {eval}`dp.fmt_pct(top10_first)` in
{eval}`P_FIRST`; since {eval}`P_STABLE` it has stayed between {eval}`dp.fmt_pct(top10_lo)` and
{eval}`dp.fmt_pct(top10_hi)`, and it was {eval}`dp.fmt_pct(top10_recent)` in {eval}`P_RECENT` against
{eval}`dp.fmt_pct(top10_prev)` in {eval}`P_PREV` ([](#fig-venues-top10)). The most-used venues of
{eval}`P_RECENT` were {eval}`recent_leaders`. Counts of distinct venues grow with the number of papers, so a
fairer comparison draws the same number of papers from each period: among {eval}`str(RAREFY_N)` papers drawn
at random, the expected number of distinct venues rose from {eval}`dp.fmt_int(rare_base)` in {eval}`P_BASE` to
{eval}`dp.fmt_int(rare_prev)` in {eval}`P_PREV` and fell to {eval}`dp.fmt_int(rare_recent)` in
{eval}`P_RECENT`. At the size of {eval}`P_PREV` ({eval}`dp.fmt_int(papers_prev)` papers), the
{eval}`P_RECENT` output would cover about {eval}`dp.fmt_int(venues_recent_rarefied)` venues, against
{eval}`dp.fmt_int(venues_prev)`: recent output is concentrated in fewer venues.

(venues-notes)=
## Notes & caveats

- **Publisher groups.** Scopus publisher names were mapped to groups by name patterns and grouped by their
  current owner: Springer Nature includes Springer, Kluwer Academic Publishers, Nature and BioMed Central;
  Elsevier includes Academic Press; Wiley includes Blackwell and Hindawi ({eval}`dp.fmt_int(n_hindawi)`
  records, among them the EURASIP journals that Hindawi published); Taylor & Francis includes Routledge and
  CRC Press; IET includes the IEE. IET journals count as IET also where Scopus names Wiley, the IET's
  publishing partner ({eval}`dp.fmt_int(n_iet_wiley)` records).
- **Missing publishers.** Of the records without a Publisher value, {eval}`dp.fmt_pct(missing_in_gap)` date
  from {eval}`GAP_TEXT`, when {eval}`dp.fmt_pct(share_pub_missing_gap)` of the records lack it. They were assigned,
  in this order, from the DOI prefix, which identifies the publisher (for example 10.1109 for IEEE, 10.1016
  for Elsevier), from the venue name ("IEEE" in a conference name; series such as
  Lecture Notes in Computer Science, CEUR Workshop Proceedings or Proceedings of SPIE), and from the
  journal's other records when these agree. Checked on the records that do have a Publisher value, the DOI
  rule gives the same group for {eval}`dp.fmt_pct(doi_agree)` of {eval}`dp.fmt_int(n_doi_check)` records and
  the venue-name rule for {eval}`dp.fmt_pct(name_agree)` of {eval}`dp.fmt_int(n_name_check)`. The
  {eval}`dp.fmt_int(n_unknown)` records that remain unknown have no DOI and were published between
  {eval}`str(unk_first)` and {eval}`str(unk_last)`; {eval}`dp.fmt_int(n_unk_conf)` of them are conference papers.
- **Journals and conferences.** The split follows the Scopus document type, as on the
  [Research output](output.ipynb) page; Scopus sometimes types papers in proceedings volumes as articles and
  papers in journal special issues as conference papers, and they are counted where Scopus puts them.
  Journals are identified by their Scopus source title; titles that share an ISSN (spelling variants and
  renamed journals that kept an ISSN) count as one journal ({eval}`dp.fmt_int(n_merged_titles)` titles merged),
  listed under its latest title. Renamed journals that share no ISSN with their former title count as separate
  journals, such as the IEE Proceedings and the IET journals that continued them.
- **Conference series.** Edition names were reduced to a series by matching the conference acronym
  ("EuCAP 2019", "ICIP'02"), by removing years, ordinals, edition numbers and "Proceedings of", and with a
  list of aliases for series whose names vary between editions (host cities in IEEE PowerTech or the AES
  Convention, renamed meetings such as the Metamaterials congress, parts of a joint meeting such as QELS at
  CLEO) and for conferences that share an acronym with another series. A conference's own workshop
  proceedings count with it (ICC Workshops, for example), while a workshop or conference held in conjunction
  with another meeting is a series of its own.
  Where Scopus gives no conference name, the series is taken from the source title
  if it names a meeting (such as a conference, symposium, congress or workshop) and is not a generic series like
  Lecture Notes in Computer Science or CEUR Workshop Proceedings. The {eval}`dp.fmt_int(n_no_series)` other
  conference papers without a conference name appeared in a journal or in a generic series; they cannot be
  assigned to a series and count under their source title in the venue counts. The power, communications and antennas groups of the
  leading series were assigned by hand from the series names.
- **Citations.** Scopus "Cited by" counts on {eval}`AS_OF_TEXT`, including self-citations. The normalised
  citation score divides each paper's citations by the mean of the School's papers of the same year and
  type (articles and reviews together), as on the [Citation impact](impact.ipynb) page; it is not a field-normalised indicator
  against world averages, and a venue's mean is sensitive to single highly cited papers.
- **Associations, not effects.** Differences between publishers or venues reflect research field, team size,
  collaboration and open-access status as much as the venue itself.
- **Diversity.** The number of distinct venues grows with the number of papers. Rarefaction, the expected
  number of distinct venues in a random draw of a fixed number of papers (Hurlbert, 1971), removes this size
  effect when periods are compared.
- **Partial year.** {eval}`str(PARTIAL)` (records to {eval}`AS_OF_TEXT`) is included in totals, in the tables
  and in the yearly chart, but not in the period comparisons.